<a href="https://colab.research.google.com/github/Ghanwa-Siddiqui/aquafina-yolo-detector/blob/main/notebooks/02_train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Train YOLOX-S on Colab
GPU work happens here. Weights, checkpoints, logs and run metadata persist in
Google Drive. Run cells deliberately; full training and weight downloading
have explicit switches. No Ultralytics package, CLI, trainer or weights.


## Repository and Drive
In Colab choose a GPU runtime with Python 3.10–3.12. Upload a ZIP containing
this repository's code, notebooks, configs, requirements and tests, and extract
it to `/content/aquafina-yolo-detector` using the Files pane. Do not include
datasets, checkpoints, caches, or `.git`. This works without a commit or push.
After an approved future push, cloning your own repository is also an option.
The repository on your computer remains the authoritative code copy.


In [1]:
!nvidia-smi

Mon Sep  7 04:10:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/aquafina-yolo")

print("Dataset:", (BASE / "processed/temple").exists())
print("Checkpoint:", (BASE / "runs/baseline-v1/latest_ckpt.pth").exists())

Dataset: True
Checkpoint: True


In [4]:
from pathlib import Path
import subprocess, sys, shutil, os

REPO = Path("/content/aquafina-yolo-detector")

subprocess.run([
    "git", "clone",
    "https://github.com/Ghanwa-Siddiqui/aquafina-yolo-detector.git",
    str(REPO)
], check=True)

print("Repository ready")

Repository ready


In [1]:
import sys
from pathlib import Path
import numpy as np
import torch

REPO = Path("/content/aquafina-yolo-detector")
YOLOX = Path("/content/YOLOX")

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(YOLOX))

import yolox

print("YOLOX:", yolox.__file__)
print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

YOLOX: /content/YOLOX/yolox/__init__.py
NumPy: 1.26.4
Torch: 2.5.1+cu121
CUDA: True
GPU: Tesla T4


In [5]:
from pathlib import Path
import torch

RUN = Path("/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1")
CKPT = RUN / "latest_ckpt.pth"

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)
print("Saved epoch:", checkpoint.get("start_epoch", checkpoint.get("epoch")))

Saved epoch: 89


In [2]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive", force_remount=True)

PREPARED = Path("/content/drive/MyDrive/aquafina-yolo/processed/temple")
LOCAL_DATA = Path("/content/temple_prepared")
RUN = Path("/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1")

assert PREPARED.exists(), "Shared processed dataset nahi mila"
assert (RUN / "latest_ckpt.pth").exists(), "Epoch checkpoint nahi mila"

if not LOCAL_DATA.exists():
    print("Dataset local disk par copy ho raha hai...")
    shutil.copytree(PREPARED, LOCAL_DATA)

DATA = LOCAL_DATA

print("Dataset ready:", DATA.exists())
print("Checkpoint ready:", (RUN / "latest_ckpt.pth").exists())

Mounted at /content/drive
Dataset local disk par copy ho raha hai...
Dataset ready: True
Checkpoint ready: True


In [3]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/aquafina-yolo-detector")
YOLOX = Path("/content/YOLOX")

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{YOLOX}"

subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", "8",
    "--resume",
], env=env, check=True)

CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'aquafina_detector.train', '--data', '/content/temple_prepared', '--run', '/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1', '--checkpoint', '/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1/latest_ckpt.pth', '--batch-size', '8', '--resume'], returncode=0)

In [4]:
import torch

checkpoint = torch.load(
    RUN / "latest_ckpt.pth",
    map_location="cpu",
    weights_only=False
)

print("Final saved epoch:", checkpoint.get("start_epoch", checkpoint.get("epoch")))
print("Best model exists:", (RUN / "best_ckpt.pth").exists())
print("Latest model exists:", (RUN / "latest_ckpt.pth").exists())

Final saved epoch: 100
Best model exists: True
Latest model exists: True


In [8]:
from pathlib import Path
import shutil

YOLOX = Path("/content/YOLOX")

if YOLOX.exists():
    shutil.rmtree(YOLOX)

print("Temporary YOLOX removed:", not YOLOX.exists())

Temporary YOLOX removed: True


In [9]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/aquafina-yolo-detector")
YOLOX = Path("/content/YOLOX")

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{YOLOX}"

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "aquafina_detector.bootstrap",
        "--repo",
        str(REPO),
    ],
    env=env,
    capture_output=True,
    text=True,
)

print("Exit code:", result.returncode)
print((result.stdout + "\n" + result.stderr)[-1500:])

Exit code: 1
 >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tensorflow 2.20.0 requires tensorboard~=2.20.0, but you have tensorboard 2.18.0 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/aquafina-yolo-detector/src/

In [9]:
import sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/aquafina-yolo-detector")
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, "/content/YOLOX")

import numpy as np
import torch

print("NumPy:", np.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

Mounted at /content/drive
NumPy: 2.0.2
CUDA: True
GPU: Tesla T4


In [2]:
from pathlib import Path
import numpy as np
import torch

print("NumPy:", np.__version__)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CKPT = Path("/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1/latest_ckpt.pth")
print("Checkpoint exists:", CKPT.exists())

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)
saved_epoch = checkpoint.get("start_epoch", checkpoint.get("epoch", "unknown"))

print("Saved epoch:", saved_epoch)

NumPy: 1.26.4
Torch: 2.5.1+cu121
CUDA: True
GPU: Tesla T4
Checkpoint exists: True
Saved epoch: 43


In [3]:
from pathlib import Path
import torch
import gc

RUN = Path("/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1")

for path in sorted(RUN.glob("*ckpt.pth")):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    epoch = ckpt.get("start_epoch", ckpt.get("epoch", "unknown"))
    print(path.name, "→ saved epoch:", epoch)
    del ckpt
    gc.collect()

best_ckpt.pth → saved epoch: 42
last_epoch_ckpt.pth → saved epoch: 43
latest_ckpt.pth → saved epoch: 43


In [4]:
from pathlib import Path
import shutil

PREPARED = Path("/content/drive/MyDrive/aquafina-yolo/processed/temple")
LOCAL_DATA = Path("/content/temple_prepared")

if not LOCAL_DATA.exists():
    print("Dataset copy ho raha hai...")
    shutil.copytree(PREPARED, LOCAL_DATA)

DATA = LOCAL_DATA
RUN = Path("/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1")

print("Dataset ready:", DATA.exists())

Dataset copy ho raha hai...
Dataset ready: True


In [5]:
import sys
import subprocess

BATCH = 8

assert DATA.exists(), "Local dataset nahi mila"
assert (RUN / "latest_ckpt.pth").exists(), "Checkpoint nahi mila"

subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", str(BATCH),
    "--resume",
], check=True)

CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'aquafina_detector.train', '--data', '/content/temple_prepared', '--run', '/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1', '--checkpoint', '/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1/latest_ckpt.pth', '--batch-size', '8', '--resume']' returned non-zero exit status 1.

In [7]:
import os
import sys
import subprocess
from pathlib import Path

REPO = Path("/content/aquafina-yolo-detector")
YOLOX = Path("/content/YOLOX")
DATA = Path("/content/temple_prepared")
RUN = Path("/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1")
BATCH = 8

assert (YOLOX / "yolox").exists(), "YOLOX folder missing"
assert DATA.exists(), "Local dataset missing"
assert (RUN / "latest_ckpt.pth").exists(), "Checkpoint missing"

env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{YOLOX}"

subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", str(BATCH),
    "--resume",
], env=env, check=True)

CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'aquafina_detector.train', '--data', '/content/temple_prepared', '--run', '/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1', '--checkpoint', '/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1/latest_ckpt.pth', '--batch-size', '8', '--resume']' returned non-zero exit status 1.

In [8]:
result = subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", "8",
    "--resume",
], env=env, capture_output=True, text=True)

print("Exit code:", result.returncode)
print((result.stdout + "\n" + result.stderr)[-4000:])

Exit code: 1

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/aquafina-yolo-detector/src/aquafina_detector/train.py", line 87, in <module>
    main()
  File "/content/aquafina-yolo-detector/src/aquafina_detector/train.py", line 83, in main
    run(a.data, a.run, a.checkpoint, a.batch_size, a.epochs, a.resume, a.workers, a.stop_after_epochs)
  File "/content/aquafina-yolo-detector/src/aquafina_detector/train.py", line 23, in run
    run_dir, checkpoint = require_drive(run_dir), require_drive(checkpoint)
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/content/aquafina-yolo-detector/src/aquafina_detector/common.py", line 43, in require_drive
    raise ValueError(f"Use a path under mounted {root}: {path}")
ValueError: Use a path under mounted /content/drive/MyDrive: /content/drive/.shortcut-targets-by-id/1Jbohj5XaIosmj_OOrOUdh02-RfZWg2N3/aquafina-yolo/runs/baseline-v1



In [9]:
from pathlib import Path
import shutil

SHARED_RUN = Path(
    "/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1"
)

RUN = Path(
    "/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1"
)

RUN.parent.mkdir(parents=True, exist_ok=True)

if not RUN.exists():
    print("Checkpoint folder copy ho raha hai...")
    shutil.copytree(SHARED_RUN, RUN)

print("New run:", RUN)
print("Checkpoint:", (RUN / "latest_ckpt.pth").exists())
print("Resolved path:", RUN.resolve())

Checkpoint folder copy ho raha hai...
New run: /content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1
Checkpoint: True
Resolved path: /content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1


In [10]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{YOLOX}"

subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", "8",``
    "--resume",
], env=env, check=True)

CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'aquafina_detector.train', '--data', '/content/temple_prepared', '--run', '/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1', '--checkpoint', '/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1/latest_ckpt.pth', '--batch-size', '8', '--resume']' returned non-zero exit status 1.

In [13]:
from pathlib import Path
import torch

RUN = Path("/content/drive/MyDrive/aquafina-yolo-training/runs/baseline-v1")
CKPT = RUN / "latest_ckpt.pth"
LOG = RUN / "train_log.txt"

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)
saved_epoch = checkpoint.get("start_epoch", checkpoint.get("epoch", "unknown"))

print("Latest saved epoch:", saved_epoch)
print("\n--- LAST LOG LINES ---")

if LOG.exists():
    lines = LOG.read_text(errors="ignore").splitlines()
    for line in lines[-30:]:
        print(line)
else:
    print("train_log.txt nahi mila")

Latest saved epoch: 89

--- LAST LOG LINES ---
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.968
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.895
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.528
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.650
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.806
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.496
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.790
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.790
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.680
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.727
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.854

2026-09-06 20:35:40.632 | INFO     | yolox.core.trainer:after_iter:253 -

In [12]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{REPO / 'src'}:{YOLOX}"

subprocess.run([
    sys.executable,
    "-u",
    "-m",
    "aquafina_detector.train",
    "--data", str(DATA),
    "--run", str(RUN),
    "--checkpoint", str(RUN / "latest_ckpt.pth"),
    "--batch-size", "8",
    "--resume",
], env=env, check=True)

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import torch

CKPT = Path("/content/drive/MyDrive/aquafina-yolo/runs/baseline-v1/latest_ckpt.pth")

assert CKPT.exists(), "Checkpoint nahi mila"

checkpoint = torch.load(CKPT, map_location="cpu", weights_only=False)

print("Saved epoch:", checkpoint.get("start_epoch", checkpoint.get("epoch", "unknown")))

In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/aquafina-yolo-detector')
DRIVE = Path('/content/drive/MyDrive/aquafina-yolo')
assert (REPO / 'pyproject.toml').is_file(), 'Extract repository code first'
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':/content/YOLOX'
PREPARED = DRIVE / 'processed/temple'


## Install the pinned GPU environment
This downloads official YOLOX **source** and Python dependencies only.
If Colab reports that already-imported packages changed, restart the session,
rerun the Repository and Drive cell, then continue with the audit cell.
Do not import torch/numpy before this install cell in a fresh session.


In [ ]:
subprocess.run([sys.executable, '-m', 'aquafina_detector.bootstrap', '--repo', str(REPO)], check=True)


In [ ]:
sys.path.insert(0, '/content/YOLOX')
import torch
from aquafina_detector.bootstrap import audit_runtime
from aquafina_detector.common import write_json
audit = audit_runtime()
assert torch.__version__.split('+')[0] == '2.5.1'
assert torch.cuda.is_available(), 'Select a GPU runtime'
write_json(DRIVE / 'environment/audit.json', audit)
print(torch.cuda.get_device_name(0), torch.__version__)


## Verify actual negative and mixed GPU batches
Synthetic fixtures test loading, Mosaic, class mapping and finite backward
passes. These are software tests, not a training dataset or accuracy evidence.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', '-m', 'gpu'], check=True)


## Official pretrained weights — explicit download to Drive
Set DOWNLOAD_WEIGHTS only when ready. Existing weights are reused with their
provenance sidecar. Never use Ultralytics checkpoints or generic `yolo` commands.


In [ ]:
from aquafina_detector.bootstrap import download_pretrained
WEIGHTS = DRIVE / 'weights/pretrained/yolox_s.pth'
DOWNLOAD_WEIGHTS = False
if DOWNLOAD_WEIGHTS:
    download_pretrained(WEIGHTS)
assert WEIGHTS.is_file(), 'Enable the explicit download once, then reuse the weights'


## Dataset and checkpoint adaptation check
DATA defaults to prepared files on Drive. For faster reads, optionally copy
PREPARED to `/content/aquafina-data` using shutil.copytree and set DATA there.
Only temporary Colab storage may hold this copy; checkpoints still use Drive.


In [ ]:
DATA = PREPARED
from yolox.utils import load_ckpt
from aquafina_detector.experiment import AquafinaExp
original = torch.load(WEIGHTS, map_location='cpu', weights_only=False)['model']
model = AquafinaExp().get_model()
mismatched = [k for k, v in model.state_dict().items() if k in original and v.shape != original[k].shape]
assert mismatched and all('cls_preds' in k for k in mismatched), mismatched
load_ckpt(model, original)
assert model.head.num_classes == 1
del model, original
print('One-class adaptation verified:', mismatched)


## One-epoch smoke run, then resume the second epoch
Set RUN_SMOKE=True after preparing data. The two commands share a two-epoch
schedule; stopping after epoch one exercises a real checkpoint resume.
Use a new SMOKE name for a repeat. Check logs for finite losses.


In [ ]:
RUN_SMOKE = False
SMOKE = DRIVE / 'runs/smoke-v1'
if RUN_SMOKE:
    command = [sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
               '--run', str(SMOKE), '--batch-size', '2', '--epochs', '2']
    subprocess.run(command + ['--checkpoint', str(WEIGHTS), '--stop-after-epochs', '1'], check=True)
    saved = torch.load(SMOKE / 'latest_ckpt.pth', map_location='cpu', weights_only=False)
    assert saved['start_epoch'] == 1 and 'scaler' in saved and 'training_model' in saved
    del saved
    subprocess.run(command + ['--checkpoint', str(SMOKE / 'latest_ckpt.pth'), '--resume'], check=True)
    saved = torch.load(SMOKE / 'latest_ckpt.pth', map_location='cpu', weights_only=False)
    assert saved['start_epoch'] == 2
    del saved
    print('Checkpoint save and resume passed')


## Baseline
Set RUN_BASELINE=True after smoke verification. Start at batch 8; if CUDA runs
out of memory, use a NEW run name and batch 4, then 2. Learning rate scales
with batch size. Do not change batch/data/schedule while resuming.


In [ ]:
RUN_BASELINE = False
RUN = DRIVE / 'runs/baseline-v1'
BATCH = 8
if RUN_BASELINE:
    subprocess.run([sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
                    '--run', str(RUN), '--checkpoint', str(WEIGHTS), '--batch-size', str(BATCH)], check=True)


## Resume after disconnection
Remount Drive, restore repository/source and environment, then enable this
cell. Use the same RUN, BATCH and DATA as the original baseline. Checkpoints
are saved each epoch; work since the last completed epoch is lost.


In [ ]:
RESUME_BASELINE = False
if RESUME_BASELINE:
    subprocess.run([sys.executable, '-m', 'aquafina_detector.train', '--data', str(DATA),
                    '--run', str(RUN), '--checkpoint', str(RUN / 'latest_ckpt.pth'),
                    '--batch-size', str(BATCH), '--resume'], check=True)
